# MedVQA — Training Walkthrough
## Fine-tuning BioViL-T + Mistral-7B with QLoRA

This notebook walks through the complete training pipeline:
1. Configuration and reproducibility setup
2. Model initialization with QLoRA
3. Training loop with custom loss
4. Evaluation and checkpointing

In [ ]:
import sys
from pathlib import Path

import torch
import wandb

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

from src.data.loader import VQARADDataset, collate_fn, create_dataloaders
from src.data.preprocessor import MedicalImagePreprocessor, TextPreprocessor
from src.models.vision_encoder import BioViLTEncoder
from src.models.language_model import MistralQLoRA
from src.models.fusion import CrossAttentionFusion
from src.models.medvqa_model import MedVQAModel
from src.training.losses import MedVQALoss
from src.training.callbacks import get_training_callbacks
from src.utils.config import MedVQAConfig
from src.utils.reproducibility import set_seed, print_system_info

print('Libraries loaded successfully')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# 1. Load Configuration
config = MedVQAConfig.from_yaml('../configs/default_config.yaml')
print('=' * 50)
print('Training Configuration')
print('=' * 50)
print(f'Model: {config.model.lm_model_name}')
print(f'Vision: {config.model.vision_encoder_name}')
print(f'Learning rate: {config.training.learning_rate}')
print(f'Batch size: {config.training.batch_size}')
print(f'Gradient accumulation: {config.training.gradient_accumulation_steps}')
print(f'Effective batch: {config.training.batch_size * config.training.gradient_accumulation_steps}')
print(f'Max epochs: {config.training.max_epochs}')
print(f'Mixed precision: bf16={config.training.bf16}, fp16={config.training.fp16}')
print(f'Seed: {config.training.seed}')
print('=' * 50)

# Set seed
set_seed(config.training.seed)
print_system_info()

In [ ]:
# 2. Setup Data
image_processor = MedicalImagePreprocessor(image_size=config.data.image_size)
text_processor = TextPreprocessor(
    model_name=config.data.tokenizer_name,
    max_question_length=config.data.max_question_length,
    max_answer_length=config.data.max_answer_length,
)

train_loader, val_loader, test_loader = create_dataloaders(
    data_dir=config.data.raw_dir,
    image_transform=image_processor,
    text_preprocessor=text_processor,
    batch_size=config.data.batch_size,
    num_workers=config.data.num_workers,
)

# Quick sanity check
batch = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  Images: {batch["images"].shape}')
print(f'  Input IDs: {batch["input_ids"].shape}')
print(f'  Labels: {batch["labels"].shape}')
print(f'  Is Yes/No: {batch["is_yesno"]}')

In [ ]:
# 3. Build Model Components
print('=' * 50)
print('Building MedVQA Model')
print('=' * 50)

# Vision encoder
vision_encoder = BioViLTEncoder(
    model_name=config.model.vision_encoder_name,
    fallback_model_name=config.model.vision_encoder_fallback,
    hidden_size=config.model.vision_hidden_size,
    projection_dim=config.model.projection_dim,
    freeze_encoder=config.model.freeze_vision_encoder,
)
print(f'Vision encoder trainable: {vision_encoder.get_trainable_params():,}')

# Language model
language_model = MistralQLoRA(
    model_name=config.model.lm_model_name,
    load_in_4bit=config.model.load_in_4bit,
    lora_r=config.model.lora_r,
    lora_alpha=config.model.lora_alpha,
    lora_dropout=config.model.lora_dropout,
    lora_target_modules=list(config.model.lora_target_modules),
    gradient_checkpointing=config.training.gradient_checkpointing,
)
language_model.print_trainable_params()

# Fusion
fusion = CrossAttentionFusion(
    d_model=config.model.projection_dim,
    n_heads=config.model.fusion_num_heads,
    dropout=config.model.fusion_dropout,
    use_residual=config.model.fusion_use_residual,
)

# Full model
medvqa_model = MedVQAModel(
    vision_encoder=vision_encoder,
    language_model=language_model,
    fusion=fusion,
    freeze_vision=config.model.freeze_vision_encoder,
    num_beams=config.model.num_beams,
)
medvqa_model.print_param_summary()

In [ ]:
# 4. Forward Pass Test
print('Testing forward pass...')
outputs = medvqa_model(
    images=batch['images'].to(device),
    input_ids=batch['input_ids'].to(device),
    attention_mask=batch['attention_mask'].to(device),
    labels=batch['labels'].to(device),
)

print(f'Forward pass logits shape: {outputs["logits"].shape}')
print(f'Loss: {outputs["loss"]}')
print()

# Test generation
print('Testing generation...')
generated = medvqa_model.generate(
    images=batch['images'][:1].to(device),
    input_ids=batch['input_ids'][:1].to(device),
    attention_mask=batch['attention_mask'][:1].to(device),
)
pred = text_processor.decode_answer(generated[0])
print(f'Question: {batch["questions"][0]}')
print(f'Ground truth: {batch["answers"][0]}')
print(f'Prediction: {pred}')
print('\n✅ Forward pass and generation working!')

In [ ]:
# 5. Setup Loss and Trainer
from transformers import TrainingArguments
from src.training.trainer import MedVQATrainer

# Loss
loss_fn = MedVQALoss(
    closed_ended_alpha=config.training.closed_ended_alpha,
    label_smoothing=config.training.label_smoothing,
    use_contrastive=config.training.use_contrastive_loss,
)

# Training arguments
training_args = TrainingArguments(
    output_dir='../experiments/medvqa_demo',
    num_train_epochs=1,  # Quick demo
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=config.training.learning_rate,
    warmup_ratio=config.training.warmup_ratio,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=20,
    save_steps=100,
    save_total_limit=2,
    bf16=config.training.bf16,
    gradient_checkpointing=config.training.gradient_checkpointing,
    report_to='none',  # Disable W&B for quick demo
    remove_unused_columns=False,
    dataloader_drop_last=True,
)

# Trainer
trainer = MedVQATrainer(
    model=medvqa_model,
    args=training_args,
    loss_fn=loss_fn,
    train_dataset=train_loader.dataset if False else None,  # Would pass dataset
    eval_dataset=val_loader.dataset if False else None,
    tokenizer=text_processor,
    data_collator=collate_fn,
    callbacks=get_training_callbacks(),
)

print('✅ Trainer configured')
print(f'Expected training steps per epoch: {len(train_loader)}')

In [ ]:
# 6. GPU Memory Estimation
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {total_mem:.1f} GB')
    print(f'Currently allocated: {allocated:.2f} GB')
    print(f'Reserved: {reserved:.2f} GB')
    print(f'Available: {total_mem - allocated:.2f} GB')
    
    # Estimate remaining capacity
    model_size = sum(p.numel() * 2 for p in medvqa_model.parameters() if p.requires_grad) / 1e9
    print(f'Estimated trainable model footprint: {model_size:.2f} GB (bf16)')
else:
    print('CUDA not available. Running in CPU mode (will be very slow).')

In [ ]:
# 7. Start Training (commented out to prevent accidental runs)
# print('Starting training...')
# trainer.train()
# print('Training complete!')
# 
# # Save model
# trainer.save_model('../experiments/medvqa_demo/final_model')
# print('Model saved!')

print('To train, uncomment the code above.')
print('Or run from terminal: python train.py --config ../configs/default_config.yaml')

## Training Commands

```bash
# Full training
python train.py --config configs/default_config.yaml

# Debug mode (1 epoch, small subset)
python train.py --config configs/default_config.yaml --debug

# Resume from checkpoint
python train.py --config configs/default_config.yaml --resume_from_checkpoint experiments/medvqa_run/checkpoint-1000
```

After training, proceed to `04_analysis.ipynb` for results and Grad-CAM visualizations.